# 3 · Search

⏱ about 13 minutes

An **embedding** turns a text into a long list of numbers, a vector, so that texts with similar
*meaning* get similar vectors, even when they share no words.

![Two sentences about cats point the same way, a sentence about revenue points elsewhere](img/embeddings.svg)

In [1]:
import os

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
CHAT_MODEL = os.environ["CHAT_MODEL"]
EMBED_MODEL = os.environ["EMBED_MODEL"]
llm = OpenAI(base_url=os.environ["LLM_BASE_URL"], api_key=os.environ["LLM_API_KEY"])
import numpy as np


def embed(texts):
    response = llm.embeddings.create(model=EMBED_MODEL, input=texts)
    return [item.embedding for item in response.data]


def similarity(a, b):
    a, b = np.array(a), np.array(b)
    return a @ b / (np.linalg.norm(a) * np.linalg.norm(b))


cat, kitten, revenue = embed([
    "The cat sits on the mat.",
    "A kitten is resting on a rug.",
    "Quarterly revenue grew by four percent.",
])
print("numbers per vector:", len(cat))
print("cat vs kitten: ", round(similarity(cat, kitten), 3))
print("cat vs revenue:", round(similarity(cat, revenue), 3))

numbers per vector: 4096
cat vs kitten:  0.574
cat vs revenue: 0.133


Try your own sentences in the cell above. Then embed all chunks from notebook 2, in batches of 64.

In [2]:
import json

chunks = json.load(open("chunks.json"))
for start in range(0, len(chunks), 64):
    batch = chunks[start : start + 64]
    for item, vector in zip(batch, embed([c["text"] for c in batch])):
        item["vector"] = vector

print(len(chunks), "chunks embedded")

197 chunks embedded


## Qdrant

Comparing a question with every chunk one by one works for 200 chunks, not for millions. A
**vector database** stores vectors and finds the closest ones fast. We use [Qdrant](https://qdrant.tech).
`":memory:"` runs it inside Python with nothing to install; in production it runs as a server.

A **collection** is like a table. Each **point** is one chunk: its vector, plus a **payload**
with everything we want back (text, file, page).

![A Qdrant collection: one point per chunk, each with id, vector and payload](img/qdrant.svg)

In [3]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, PointStruct, VectorParams

db = QdrantClient(":memory:")
db.create_collection(
    "papers",
    vectors_config=VectorParams(size=len(chunks[0]["vector"]), distance=Distance.COSINE),
)
db.upsert(
    "papers",
    points=[
        PointStruct(id=i, vector=c["vector"], payload={k: c[k] for k in ("source", "page", "text")})
        for i, c in enumerate(chunks)
    ],
)

question = "What were the lifetime-encoded beads loaded with?"
hits = db.query_points("papers", query=embed([question])[0], limit=5).points
for hit in hits:
    print(f"{hit.score:.3f}  {hit.payload['source']} p. {hit.payload['page']}")
    print("      ", hit.payload["text"][:150].replace("\n", " "), "\n")

0.734  Kage_2018_SciReports.pdf p. 2
        extend the accessible lifetime range. The beads serve as a model system in our studies but further applications will not be limited to distinguishing 

0.639  Kage_2018_SciReports.pdf p. 2
       S core/shell quantum dots (QDs) and a final poly(sodium 4-styrensulfonate) (PSS) layer provided by the research group of Sukhanova and Nabiev 34,35 .  

0.636  Kage_2018_SciReports.pdf p. 3
       Table 1. Lifetime codes and respective luminophores with measurement parameters, mean bead diameter D and obtained ensemble (reference) lifetimes calc 

0.631  Kage_2018_SciReports.pdf p. 4
       8 . Details are given in the Supplementary Information. Repeated simulations were carried out for each parameter set under study to obtain a reasonabl 

0.622  Kage_2018_SciReports.pdf p. 4
       or lifetime encoding had to be excitable at a single common wavelength (here: 488 nm) and detectable within a certain spectral window. Moreover, they  



The chatbot runs as a separate program, so it can't see this notebook's variables. We save the
chunks with their vectors, and the bot loads them into its own Qdrant.

In [4]:
from pathlib import Path

Path("index.json").write_text(json.dumps(chunks))
print("saved", len(chunks), "chunks to index.json")

saved 197 chunks to index.json


## A search engine in Chainlit

No LLM yet: the bot shows the five closest chunks and their scores.

In [5]:
%%writefile app.py
import json
import os

import chainlit as cl
from dotenv import load_dotenv
from openai import AsyncOpenAI
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, PointStruct, VectorParams

load_dotenv()
llm = AsyncOpenAI(base_url=os.environ["LLM_BASE_URL"], api_key=os.environ["LLM_API_KEY"])

chunks = json.load(open("index.json"))
db = QdrantClient(":memory:")
db.create_collection(
    "papers",
    vectors_config=VectorParams(size=len(chunks[0]["vector"]), distance=Distance.COSINE),
)
db.upsert(
    "papers",
    points=[
        PointStruct(id=i, vector=c["vector"], payload={k: c[k] for k in ("source", "page", "text")})
        for i, c in enumerate(chunks)
    ],
)

# TODO 2: a dropdown to pick the paper to search in.


async def search(question, limit=5):
    response = await llm.embeddings.create(model=os.environ["EMBED_MODEL"], input=[question])
    # TODO 1: fewer results, and only good ones.
    return db.query_points("papers", query=response.data[0].embedding, limit=limit).points


@cl.on_message
async def on_message(message: cl.Message):
    hits = await search(message.content)
    results = [
        f"**{h.payload['source']}, p. {h.payload['page']}** (score {h.score:.2f})\n"
        f"> {h.payload['text'][:300].replace(chr(10), ' ')}…"
        for h in hits
    ]
    await cl.Message(content="\n\n".join(results) or "Nothing found.").send()

Overwriting app.py


In [6]:
import socket
import subprocess
import sys

# Start the chatbot, unless it's still running from an earlier notebook.
with socket.socket() as probe:
    running = probe.connect_ex(("127.0.0.1", 8000)) == 0
if not running:
    subprocess.Popen(
        [sys.executable, "-m", "chainlit", "run", "app.py", "--watch", "--headless"],
        stdout=open("chainlit.log", "w"),
        stderr=subprocess.STDOUT,
    )
print("Chatbot: http://localhost:8000")

Chatbot: http://localhost:8000


## Your turn

Edit the `%%writefile app.py` cell above, run it again, and try your change in the browser.

Ask a few questions in the browser, including one the papers can't answer, like
*What is the capital of France?* Search always returns something: the closest chunks, however far
away they are.

1. **Fewer, better results.** Show 3 results instead of 5, and leave out the ones that don't
   match. Compare the scores of a good question and of the France question, pick a cutoff in
   between, and pass it to `query_points` as `score_threshold=`. Ask about France again.
2. **Search one paper only.** Add a dropdown to the chat settings to pick a paper, and only search
   that one. You need:
   - `await cl.ChatSettings([Select(id="paper", label="Search in", values=[...])]).send()` in a
     function decorated with `@cl.on_chat_start` (`from chainlit.input_widget import Select`),
   - a function decorated with `@cl.on_settings_update` that receives the settings as a dict and
     stores the choice with `cl.user_session.set(...)`,
   - a filter for Qdrant: `query_filter=Filter(must=[FieldCondition(key="source", match=MatchValue(value=...))])`
     (all three from `qdrant_client.models`).
3. **Bonus: open the page.** Attach each hit as a PDF that opens at the right page:
   `cl.Pdf(name=..., path=f"data/{source}", page=page, display="side")`. Use the same text as
   name that you show in bold, so Chainlit turns it into a link.

<details>
<summary><b>Solutions</b> (try first!)</summary>

**1**
```python
async def search(question, limit=3):
    response = await llm.embeddings.create(model=os.environ["EMBED_MODEL"], input=[question])
    return db.query_points(
        "papers", query=response.data[0].embedding, limit=limit,
        score_threshold=0.3,  # real questions scored 0.33–0.73 here, France 0.17
    ).points
```

**2**
```python
from chainlit.input_widget import Select
from qdrant_client.models import FieldCondition, Filter, MatchValue

ALL = "all papers"


@cl.on_chat_start
async def on_chat_start():
    papers = sorted({c["source"] for c in chunks})
    await cl.ChatSettings([Select(id="paper", label="Search in", values=[ALL] + papers, initial_index=0)]).send()


@cl.on_settings_update
async def on_settings_update(settings):
    cl.user_session.set("paper", settings["paper"])


async def search(question, limit=5):
    response = await llm.embeddings.create(model=os.environ["EMBED_MODEL"], input=[question])
    paper = cl.user_session.get("paper") or ALL
    only = None if paper == ALL else Filter(must=[FieldCondition(key="source", match=MatchValue(value=paper))])
    return db.query_points("papers", query=response.data[0].embedding, limit=limit, query_filter=only).points
```

**3** — in `on_message`:
```python
    names = [f"{h.payload['source']}, p. {h.payload['page']}" for h in hits]
    pdfs = [
        cl.Pdf(name=name, path=f"data/{h.payload['source']}", page=h.payload["page"], display="side")
        for name, h in zip(names, hits)
    ]
    await cl.Message(content="\n\n".join(results), elements=pdfs).send()
```
</details>